# RFM Analysis
RFM adalah cara mengelompokkan pelanggan berdasarkan 3 angka sederhana:

* **Recency (R)** — sudah berapa lama sejak pelanggan itu belanja terakhir kali
* **Frequency (F)** — berapa kali pelanggan itu belanja (total transaksi)
* **Monetary (M)** — berapa total uang yang sudah dihabiskan pelanggan itu

Tujuannya: dari 3 angka ini, kita bisa kelompokkan pelanggan jadi kategori seperti “pelanggan setia & bernilai tinggi” vs “pelanggan yang mulai hilang” vs “pelanggan baru”. Ini teknik standar di dunia kerja untuk strategi marketing (misal: siapa yang perlu dikirim promo supaya tidak kabur).

**Insight:**

Segmentasi RFM terhadap 93,357 pelanggan Olist menghasilkan 3 kelompok 
dengan ukuran tidak merata:

- **Pelanggan Terbaik** (skor ≥ 8): 46,347 pelanggan (49.64%)
- **Pelanggan Biasa** (skor 4-7): 45,491 pelanggan (48.73%)
- **Pelanggan Berisiko Hilang** (skor < 4): 1,519 pelanggan (1.63%)

Hampir seluruh pelanggan (98.37%) berada di kategori Terbaik atau Biasa, 
sementara hanya 1.63% masuk kategori Berisiko Hilang.

In [11]:
import pandas as pd
import psycopg2

conn = psycopg2.connect(
    host="localhost",
    dbname="Olist Ecommerce Database",
    user="postgres",
    password="admin123",
    port=5432
)

In [12]:
query = """
SELECT 
    c.customer_unique_id,
    o.order_id,
    o.order_purchase_timestamp,
    p.payment_value
FROM olist_orders_dataset o
JOIN olist_customers_dataset c ON o.customer_id = c.customer_id
JOIN olist_order_payments_dataset p ON o.order_id = p.order_id
WHERE o.order_status = 'delivered'
"""

df = pd.read_sql(query, conn)
conn.close()

df.head()

C:\Users\alex7\AppData\Local\Temp\ipykernel_2516\3393445721.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,customer_unique_id,order_id,order_purchase_timestamp,payment_value
0,708ab75d2a007f0564aedd11139c7708,b81ef226f3fe1789b1e8b2acac839d17,2018-04-25 22:01:49,99.33
1,baf8a55609217194b2feb8adacdcc144,07d13d352c85dfbfa60dc56dea818335,2017-01-22 23:07:15,32.62
2,55020cb68ae358a51969695657a54e9c,b545ba7b0bd67a3128185c7214704319,2018-06-11 14:59:24,340.08
3,82f78e34ccf4f5e160f2b36eef6aa1e7,9d8065ae4c6f5725ee052d02ab00f64e,2017-03-30 16:36:12,82.06
4,7f0b11f47d3ff4b3f1ce8788cde95a36,d9d104af1ea99e0b7f6b0644eedc74d0,2017-02-15 00:24:14,40.95


In [13]:
# Tanggal acuan "hari ini" — karena dataset ini historis, kita pakai 
# tanggal transaksi terakhir yang ada di data sebagai acuan "sekarang"
tanggal_acuan = df['order_purchase_timestamp'].max()

rfm = df.groupby('customer_unique_id').agg(
    recency=('order_purchase_timestamp', lambda x: (tanggal_acuan - x.max()).days),
    frequency=('order_id', 'nunique'),
    monetary=('payment_value', 'sum')
).reset_index()

rfm.head()

,customer_unique_id,recency,frequency,monetary
0,0000366f3b9a7992bf8c76cfdf3221e2,111,1,141.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,114,1,27.19
2,0000f46a3911fa3c0805444483337064,536,1,86.22
3,0000f6ccb0745a6a4b88665a16c9f078,320,1,43.62
4,0004aac84e0df4da2b147fca70cf8255,287,1,196.89


In [14]:
rfm[['recency', 'frequency', 'monetary']].describe()

,recency,frequency,monetary
count,93357.000000,93357.000000,93357.000000
mean,236.936673,1.033420,165.198772
std,152.584315,0.209099,226.314579
min,0.000000,1.000000,9.590000
25%,113.000000,1.000000,63.060000
50%,218.000000,1.000000,107.780000
75%,345.000000,1.000000,182.560000
max,694.000000,15.000000,13664.080000


In [15]:
# Recency: makin KECIL angkanya (baru belanja) makin BAIK, 
# jadi urutan skornya dibalik (pakai labels menurun)
rfm['r_score'] = pd.qcut(rfm['recency'], q=4, labels=[4, 3, 2, 1])

# Frequency: makin BESAR angkanya (sering belanja) makin BAIK
# Karena banyak pelanggan nilainya sama (1), dipakai rank dulu supaya 
# tidak error saat dibagi kuartil
rfm['f_score'] = pd.qcut(rfm['frequency'].rank(method='first'), q=4, labels=[1, 2, 3, 4])

# Monetary: makin BESAR angkanya (banyak belanja) makin BAIK
rfm['monetary_score'] = pd.qcut(rfm['monetary'], q=4, labels=[1, 2, 3, 4])

rfm.head()

,customer_unique_id,recency,frequency,monetary,r_score,f_score,monetary_score
0,0000366f3b9a7992bf8c76cfdf3221e2,111,1,141.90,4,1,3
1,0000b849f77a49e4a4ce2b2a4ca5be3f,114,1,27.19,3,1,1
2,0000f46a3911fa3c0805444483337064,536,1,86.22,1,1,2
3,0000f6ccb0745a6a4b88665a16c9f078,320,1,43.62,2,1,1
4,0004aac84e0df4da2b147fca70cf8255,287,1,196.89,2,1,4


In [16]:
# Gabungkan 3 skor jadi 1 kode, misal "432" artinya r=4, f=3, m=2
rfm['rfm_score'] = (
    rfm['r_score'].astype(str) + 
    rfm['f_score'].astype(str) + 
    rfm['monetary_score'].astype(str)
)

# Buat kategori berdasarkan rentang nilai total_score (fixed threshold)
rfm['total_score'] = (
    rfm['r_score'].astype(int) + 
    rfm['f_score'].astype(int) + 
    rfm['monetary_score'].astype(int)
)

def beri_segmen(skor):
    if skor >= 8:
        return 'Pelanggan Terbaik'
    elif skor >= 4:
        return 'Pelanggan Biasa'
    else:
        return 'Pelanggan Berisiko Hilang'

rfm['segmen'] = rfm['total_score'].apply(beri_segmen)

# Tampilkan jumlah DAN persentase sekaligus
segmen_count = rfm['segmen'].value_counts()
segmen_percent = rfm['segmen'].value_counts(normalize=True) * 100

segmen_summary = pd.DataFrame({
    'jumlah': segmen_count,
    'persentase': segmen_percent.round(2)
})

segmen_summary

,jumlah,persentase
segmen,,
Pelanggan Terbaik,46347,49.64
Pelanggan Biasa,45491,48.73
Pelanggan Berisiko Hilang,1519,1.63


In [21]:
# Export hasil RFM ke CSV untuk dipakai di Power BI
rfm.to_csv('../data/processed/rfm_result.csv', index=False)
print("Berhasil disimpan ke data/processed/rfm_result.csv")

Berhasil disimpan ke data/processed/rfm_result.csv
